In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()

In [ ]:
df = q(
    """ 
SELECT ft.idx, ft.fuel_kg, ft.flight_id, flt.aircraft_type, ((ft.`end` - ft.start)/1e9) as duration_s, 
TIMESTAMP_DIFF(TIMESTAMP_MICROS(DIV(ft.start, 1000)), flt.takeoff, SECOND) AS since_takeoff_s,
TIMESTAMP_DIFF(flt.landed, TIMESTAMP_MICROS(DIV(ft.`end`, 1000)), SECOND) AS before_landing_s
FROM `prc-data-challenge-2025.prc2025.fuel_train` as ft
JOIN `prc-data-challenge-2025.prc2025.flightlist_train` as flt
ON ft.flight_id = flt.flight_id
"""
)

In [ ]:
df.describe()

131,530 intervals, no missing values, `idx` unique (join loses nothing).
Negative `since_takeoff_s` / `before_landing_s` down to about -4 h: intervals far outside the flight.
`fuel_kg` max = 32,205 kg in at most 1 h: impossible.

In [ ]:
flights = df.groupby("flight_id").agg(
    fuel_kg_sum=("fuel_kg", "sum"),
    duration_s_sum=("duration_s", "sum"),
    aircraft_type=("aircraft_type", "first"),
)

flights["fuel_rate_kg_per_s"] = flights["fuel_kg_sum"] / flights["duration_s_sum"]

11,037 flights: average fuel rate per flight, input of the Tukey rule.

In [ ]:
# Tukey far-out bounds per aircraft type and outlier flights
grouped = flights.groupby("aircraft_type")["fuel_rate_kg_per_s"]
flights["q1"] = grouped.transform(lambda s: s.quantile(0.25))
flights["q3"] = grouped.transform(lambda s: s.quantile(0.75))
flights["iqr"] = flights["q3"] - flights["q1"]
flights["lower_bound"] = flights["q1"] - 3 * flights["iqr"]
flights["upper_bound"] = flights["q3"] + 3 * flights["iqr"]

outliers = flights[
    (flights["fuel_rate_kg_per_s"] < flights["lower_bound"])
    | (flights["fuel_rate_kg_per_s"] > flights["upper_bound"])
]
len(outliers)

205 far-out flights, same as in `explore_links`.

In [ ]:
ground_overflow = df[(df["since_takeoff_s"] < -1800) | (df["before_landing_s"] < -1800)]
ground_flights = ground_overflow["flight_id"].unique()
len(ground_flights)

189 flights (410 intervals) with an interval more than 30 min before takeoff or after landing.

In [ ]:
bad_flights = set(outliers.index) | set(ground_flights)
df_train = df[~df["flight_id"].isin(bad_flights)]
len(df_train), len(df)

385 flights removed (9 in both lists, 3.5% of flights), i.e. 3,929 intervals (3%).

In [ ]:
flights = df_train["flight_id"].unique()
val_flights = pd.Series(flights).sample(frac=0.2, random_state=42)

val = df_train[df_train["flight_id"].isin(val_flights)]
fit = df_train[~df_train["flight_id"].isin(val_flights)]

In [ ]:
fit_rate = fit.fuel_kg / fit.duration_s
rate_by_type = fit_rate.groupby(fit.aircraft_type).median()

val["rate"] = val.aircraft_type.map(rate_by_type).fillna(fit_rate.median())
val["fuel_pred"] = val["rate"] * val["duration_s"]
val["sq_error"] = (val["fuel_kg"] - val["fuel_pred"]) ** 2
rmse = (val["sq_error"].mean()) ** 0.5
print(rmse)

Baseline (median rate per aircraft type x duration): RMSE 213 kg.
Too optimistic: outliers were also removed from validation.

In [ ]:
flights = df["flight_id"].unique()
val_flights = pd.Series(flights).sample(frac=0.2, random_state=42)

val = df[df["flight_id"].isin(val_flights)]
fit = df[(~df["flight_id"].isin(val_flights)) & (~df["flight_id"].isin(bad_flights))]

fit_rate = fit.fuel_kg / fit.duration_s
rate_by_type = fit_rate.groupby(fit.aircraft_type).median()

val["rate"] = val.aircraft_type.map(rate_by_type).fillna(fit_rate.median())
val["fuel_pred"] = val["rate"] * val["duration_s"]
val["sq_error"] = (val["fuel_kg"] - val["fuel_pred"]) ** 2
rmse = (val["sq_error"].mean()) ** 0.5
print(rmse)

Honest validation (outliers removed from `fit` only, kept in `val`): **RMSE 330 kg**. Reference to beat.
Training without removing outliers gives 329.9 kg: no difference, the median already ignores extremes.